In [ ]:
from importlib import reload
from os.path import join
from collections import defaultdict
import sys
import os
import numpy as np
from matplotlib import pyplot
import copy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as tf
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import torchmetrics

In [ ]:
from trackertraincode.datasets.dshdf5pose import Hdf5PoseDataset
from trackertraincode.datasets.batch import Batch
import trackertraincode.datatransformation as dtr
import trackertraincode.neuralnets.models as models
import trackertraincode.neuralnets.modelcomponents as modelcomponents
import trackertraincode.vis as vis
import trackertraincode.utils as utils
import trackertraincode.train as train
import trackertraincode.pipelines as pipelines
import trackertraincode.eval as eval

In [ ]:
%matplotlib ipympl

In [ ]:
#net = eval.load_localizer_network("../model_files/head-localizer.onnx", "cuda")
net = eval.load_localizer_network("../model_files/current/localizer-retrain/swa.ckpt", "cuda")

In [ ]:
def iterate_data():
    ds = pipelines.make_validation_dataset("wider", additional_transforms=[dtr.batch.to_numpy])
    for i in range(len(ds)):
        yield ds[i]
vis.matplotlib_plot_iterable(iterate_data(), vis.draw_dataset_sample)

In [ ]:
def iterate_predictions(loader, net : eval.InferenceNetwork):
    for batch in loader:
        normalized_batch = dtr.batch.normalize_batch(batch)
        normalized_batch = dtr.batch.whiten_batch(normalized_batch)
        preds = net(normalized_batch['image'].to(net.device_for_input))
        preds = Batch(batch.meta, **preds)
        preds = dtr.batch.unnormalize_batch(preds)
        preds = preds.to("cpu")
        for sample, pred in zip(Batch.undo_collate(batch), Batch.undo_collate(preds)):
                sample = dtr.batch.to_numpy(sample)
                pred = dtr.batch.to_numpy(pred)
                yield (sample, pred)

In [ ]:
h,w = net.input_shape
test_loader = pipelines.make_validation_loader("wider", additional_sample_transform=[dtr.batch.FocusRoi((w, h), 1., roi_variable=None)])

vis.matplotlib_plot_iterable(iterate_predictions(test_loader, net), vis.draw_prediction)

In [ ]:
def analyze(net, loader : DataLoader[Batch]):
    metrics = torchmetrics.MetricCollection({"hasface" : eval.IsFaceAccuracy(), "roi" : eval.LocalizerBoxMeanSquareErrors(0.5)})
    for batch in loader:
        normalized_batch = dtr.batch.normalize_batch(batch)
        normalized_batch = dtr.batch.whiten_batch(normalized_batch)
        preds = net(normalized_batch['image'].to(net.device_for_input))
        preds = Batch(batch.meta, **preds)
        preds = dtr.batch.unnormalize_batch(preds)
        preds = preds.to("cpu")
        metrics.update(preds, batch)
    results = metrics.compute()
    accuracies : eval.AccuracyResults = results['hasface']
    roi_mses = results['roi']
    roi_mses = roi_mses[torch.isfinite(roi_mses)]
    roi_rmse = torch.sqrt(torch.mean(roi_mses))
    print("At thresholds:")
    print(accuracies.thresholds)
    print(accuracies.accuracies)
    print (f"rel RMSE @ 50% = {roi_rmse*100:0.1f} %")

In [ ]:
analyze(net, test_loader)

Original localizer
------------------

At thresholds: `tensor([0.1000, 0.3300, 0.5000, 0.6600, 0.9000])`

Accuracies `tensor([0.5090, 0.8980, 0.9260, 0.9280, 0.8200])`

rel RMSE `@ 50% = 19.3 %`